# Notebook 2 - Verify the XML Cache

not much to see here, this notebook just checks that all the downloaded XML files are actually valid and contain real holdings data.

i added this step because early on i kept getting empty results and had no idea why. turns out sometimes you download the wrong XML file (the cover page instead of the holdings table) and it looks fine until you try to parse it and get nothing.

so now we verify:
- file actually exists on disk
- its valid XML (not a HTML error page from the SEC)
- it contains infoTable elements which means it has actual holdings

In [ ]:
import os, sys, pathlib

# ----- COLAB SETUP -----
# Option A: with Google Drive (data stays between sessions)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: without Drive (data gone when session ends, need to rerun)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# detects root automatically when running locally
PROJECT_ROOT = pathlib.Path("/content/13F-Analytics") if pathlib.Path("/content/13F-Analytics").exists() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("working directory:", PROJECT_ROOT)

In [ ]:
from pathlib import Path
from xml.etree import ElementTree as ET
from src import config
from src.utils import load_parquet
from src.sec import looks_like_info_table

filings = load_parquet(config.FILINGS_PARQUET)
filings[["quarter", "xml_file", "local_xml_path"]]

In [ ]:
report = []
for _, f in filings.iterrows():
    p = Path(f["local_xml_path"])
    txt = p.read_text(encoding="utf-8")
    ok_xml = True
    try:
        ET.fromstring(txt)
    except ET.ParseError:
        ok_xml = False
    report.append({
        "quarter": f["quarter"],
        "file": p.name,
        "size_kb": round(p.stat().st_size / 1024, 1),
        "well_formed_xml": ok_xml,
        "has_holdings": looks_like_info_table(txt),
        "is_html_error": txt.lstrip()[:6].lower().startswith("<html"),
    })

import pandas as pd
audit = pd.DataFrame(report)
audit

In [ ]:
# all of these should be True, True, False
# if has_holdings is False for any quarter that XML is the cover page not the holdings
assert audit["well_formed_xml"].all(), "malformed XML - rerun notebook 1"
assert audit["has_holdings"].all(), "cover page downloaded instead of holdings table - rerun notebook 1"
assert not audit["is_html_error"].any(), "got HTML error page, check your User-Agent in config.py"
print("all good -", len(audit), "quarters verified")